# Legendas automáticas: encoder ResNet-50

Faculdade de Ciências e Tecnologias em Engenharia - Campus UnB Gama <br>
Tópicos Especiais em Matemática Aplicada: Deep Learning <br>
Professor: Vinicius Rispoli<br>
Alunos: Eliseu Kadesh, Pedro Barbosa e Raquel Eucaria.

Treina três decoders (LSTM, GRU e Transformer) sobre o mesmo encoder ResNet-50 congelado e compara o BLEU. Contexto, resultados e limitações estão no [README](../README.md).

## 1. Configuração e dados

In [ ]:
!pip install torchmetrics
!pip install -q kagglehub

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from torchmetrics.text import BLEUScore
from PIL import Image
import os
import pandas as pd

import kagglehub

import spacy

from torch.nn.utils.rnn import pad_sequence

import time

In [ ]:
# Configuração do dispositivo (Usa GPU se disponível para ser mais rápido)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

# Definição das transformações para a imagem
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

print("Configuração concluída!")

Usando dispositivo: cuda
Configuração concluída!


### Vocabulário, dataset e encoder

O vocabulário guarda as palavras que aparecem ao menos 5 vezes. O encoder ResNet-50 congelado gera um vetor de 256 dimensões por imagem.

In [ ]:
# --- CLASSE 1: VOCABULARY (O Tradutor Texto <-> Números) ---
class Vocabulary:
    def __init__(self, freq_threshold):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.freq_threshold = freq_threshold

    def __len__(self):
        return len(self.itos)

    @staticmethod
    def tokenizer_eng(text):
        return [tok.text.lower() for tok in spacy_eng.tokenizer(str(text))]

    def build_vocabulary(self, sentence_list):
        frequencies = {}
        idx = 4
        for sentence in sentence_list:
            for word in self.tokenizer_eng(sentence):
                if word not in frequencies:
                    frequencies[word] = 1
                else:
                    frequencies[word] += 1
                if frequencies[word] == self.freq_threshold:
                    self.stoi[word] = idx
                    self.itos[idx] = word
                    idx += 1

    def numericalize(self, text):
        tokenized_text = self.tokenizer_eng(text)
        return [
            self.stoi[token] if token in self.stoi else self.stoi["<UNK>"]
            for token in tokenized_text
        ]

# --- CLASSE 2: DATASET (O Organizador dos Arquivos) ---
class FlickrDataset(Dataset):
    def __init__(self, root_dir, captions_file, transform=None, freq_threshold=5):
        self.root_dir = root_dir
        self.df = pd.read_csv(captions_file)
        self.transform = transform

        # Pega colunas de imagem e legenda
        self.imgs = self.df["image"]
        self.captions = self.df["caption"]

        # Cria o vocabulário
        self.vocab = Vocabulary(freq_threshold)
        self.vocab.build_vocabulary(self.captions.tolist())

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        caption = self.captions[index]
        img_id = self.imgs[index]

        # Carrega imagem
        img = Image.open(os.path.join(self.root_dir, img_id)).convert("RGB")
        if self.transform:
            img = self.transform(img)

        # Processa legenda
        numericalized_caption = [self.vocab.stoi["<SOS>"]]
        numericalized_caption += self.vocab.numericalize(caption)
        numericalized_caption.append(self.vocab.stoi["<EOS>"])

        return img, torch.tensor(numericalized_caption)

# --- CLASSE 3: COLLATE (O Ajuste de Tamanho) ---
class MyCollate:
    def __init__(self, pad_idx):
        self.pad_idx = pad_idx

    def __call__(self, batch):
        imgs = [item[0].unsqueeze(0) for item in batch]
        imgs = torch.cat(imgs, dim=0)
        targets = [item[1] for item in batch]
        targets = pad_sequence(targets, batch_first=True, padding_value=self.pad_idx)
        return imgs, targets

# --- CLASSE 4: ENCODER (A Visão do Modelo - ResNet) ---
class EncoderCNN(nn.Module):
    def __init__(self, embed_size, train_CNN=False):
        super(EncoderCNN, self).__init__()



        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

        modules = list(resnet.children())[:-1] # Remove a última camada
        self.resnet = nn.Sequential(*modules)
        self.embed = nn.Linear(resnet.fc.in_features, embed_size)
        self.bn = nn.BatchNorm1d(embed_size, momentum=0.01)

        # Congelar pesos da ResNet
        for param in self.resnet.parameters():
            param.requires_grad = train_CNN

    def forward(self, images):
        features = self.resnet(images)
        features = features.reshape(features.size(0), -1)
        features = self.embed(features)
        features = self.bn(features)
        return features

print("✅ Todas as Classes foram definidas!")

✅ Todas as Classes foram definidas!


### Carregamento do Flickr8k

Download via `kagglehub` ([adityajn105/flickr8k](https://www.kaggle.com/datasets/adityajn105/flickr8k)): cerca de 8,1 mil imagens, com 5 legendas cada.

In [ ]:
# 1. Baixar Dataset
print("⏳ Baixando dataset...")
path = kagglehub.dataset_download("adityajn105/flickr8k")
image_folder = os.path.join(path, "Images")
caption_file = os.path.join(path, "captions.txt")


# Carregar o modelo de linguagem inglês do spaCy
spacy_eng = spacy.load("en_core_web_sm")

# 2. Instanciar Dataset e Loader
print("⚙️ Processando vocabulário...")
dataset = FlickrDataset(
    root_dir=image_folder,
    captions_file=caption_file,
    transform=transform,
    freq_threshold=5
)

pad_idx = dataset.vocab.stoi["<PAD>"]

train_loader = DataLoader(
    dataset=dataset,
    batch_size=32,
    num_workers=2,
    shuffle=True,
    collate_fn=MyCollate(pad_idx=pad_idx)
)

# 3. Teste Final
print("\n✅ Dataset carregado com sucesso!")
print(f"Tamanho do Vocabulário: {len(dataset.vocab)} palavras")

# Pegar um lote para ver se o Encoder funciona
imgs, captions = next(iter(train_loader))
imgs = imgs.to(device)

# Testar o Encoder
encoder = EncoderCNN(embed_size=256).to(device)
features = encoder(imgs)

print(f"Formato das Imagens de entrada: {imgs.shape}") # [32, 3, 224, 224]
print(f"Formato das Features (saiu do Encoder): {features.shape}") # [32, 256]
print("Tudo pronto para criar os Decoders! 🚀")

⏳ Baixando dataset...
⚙️ Processando vocabulário (isso pode levar uns segundos)...

✅ Dataset carregado com sucesso!
Tamanho do Vocabulário: 2994 palavras
Formato das Imagens de entrada: torch.Size([32, 3, 224, 224])
Formato das Features (saiu do Encoder): torch.Size([32, 256])
Tudo pronto para criar os Decoders! 🚀


## 2. Decoders

### LSTM e GRU

O vetor da imagem entra como primeiro passo da sequência, seguido das palavras da legenda (teacher forcing).

In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, num_layers, rnn_type="LSTM"):
        super(DecoderRNN, self).__init__()
        self.rnn_type = rnn_type
        self.embed = nn.Embedding(vocab_size, embed_size)

        # Escolha entre LSTM e GRU
        if rnn_type == "LSTM":
            self.rnn = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        else: # GRU
            self.rnn = nn.GRU(embed_size, hidden_size, num_layers, batch_first=True)

        self.linear = nn.Linear(hidden_size, vocab_size)
        self.dropout = nn.Dropout(0.5)

    def forward(self, features, captions):
        # 1. Cria os Embeddings das legendas
        # input: <SOS> O gato ...
        # output: O gato ... <EOS>
        embeddings = self.dropout(self.embed(captions[:, :-1]))

        # 2. Concatena a Imagem como se fosse a "primeira palavra" da frase
        # inout: [IMAGEM, <SOS>, O, gato, ...]
        inputs = torch.cat((features.unsqueeze(1), embeddings), dim=1)

        # 3. Passa pela RNN
        hiddens, _ = self.rnn(inputs)

        # 4. Gera as previsões para cada passo de tempo
        outputs = self.linear(hiddens)

        # O RNN retorna uma sequência de comprimento L (Imagem + L-1 tokens).
        return outputs[:, 1:, :]

### Transformer

Uma camada de decoder com 4 cabeças, positional encoding senoidal e máscara causal. O vetor da imagem é a memória da atenção cruzada.

In [ ]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

class DecoderTransformer(nn.Module):
    def __init__(self, embed_size, vocab_size, num_heads, num_layers):
        super(DecoderTransformer, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.pos_encoder = PositionalEncoding(embed_size)
        decoder_layer = nn.TransformerDecoderLayer(d_model=embed_size, nhead=num_heads, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.linear = nn.Linear(embed_size, vocab_size)
        self.dropout = nn.Dropout(0.5)

    def generate_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask.to(device)

    def forward(self, features, captions):
        captions_input = captions[:, :-1]
        embeddings = self.dropout(self.pos_encoder(self.embed(captions_input)))
        features = features.unsqueeze(1)
        tgt_mask = self.generate_mask(embeddings.size(1))
        output = self.transformer_decoder(tgt=embeddings, memory=features, tgt_mask=tgt_mask)
        outputs = self.linear(output)
        return outputs

## 3. Treino

Adam (lr 3e-4), 10 épocas, batch 32 e entropia cruzada ignorando `<PAD>`. O log mostra a loss média de cada época.

In [ ]:
def train_model(encoder, decoder, data_loader, criterion, optimizer, num_epochs, model_name="modelo"):
    encoder.train()
    decoder.train()
    print(f"--- Iniciando Treino: {model_name} ---")

    for epoch in range(num_epochs):
        start_time = time.time()
        total_loss = 0

        for idx, (imgs, captions) in enumerate(data_loader):
            imgs = imgs.to(device)
            captions = captions.to(device)

            optimizer.zero_grad()

            with torch.no_grad():
                features = encoder(imgs)

            outputs = decoder(features, captions)

            # CORREÇÃO: O alvo (targets) é a legenda sem o primeiro token (<SOS>).
            # A saída (outputs) do decoder já tem o tamanho correto para corresponder ao alvo.
            # A linha que fatiáva o `outputs` foi removida.
            targets = captions[:, 1:]

            # A forma da saída (outputs) e do alvo (targets) devem ser compatíveis antes do reshape.
            # Se o decoder (como o Transformer) já retorna [B, L-1, V], não é necessário fatiar.
            # Se o decoder (como o RNN) retorna [B, L, V], ele precisa ser fatiado.
            # A solução mais robusta é garantir que a saída tenha o mesmo comprimento de sequência que o alvo.
            if outputs.shape[1] != targets.shape[1]:
                 raise ValueError(f"Disparidade no comprimento da sequência! Saída: {outputs.shape[1]}, Alvo: {targets.shape[1]}")

            loss = criterion(outputs.reshape(-1, outputs.shape[2]), targets.reshape(-1))

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if idx % 100 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}], Step [{idx}/{len(data_loader)}], Loss: {loss.item():.4f}")

        avg_loss = total_loss / len(data_loader)
        elapsed = time.time() - start_time
        print(f"✅ Epoch {epoch+1} finalizada! Loss Média: {avg_loss:.4f} | Tempo: {elapsed:.0f}s")

    print(f"Treino de {model_name} concluído!")

### LSTM

In [ ]:
# --- MODELO 1: LSTM ---

# 1. Configurações
embed_size = 256
hidden_size = 256
vocab_size = len(dataset.vocab)
num_layers = 1
learning_rate = 3e-4
num_epochs = 10

# 2. Instanciar Modelo
print("🤖 Criando modelo LSTM...")
encoder = EncoderCNN(embed_size).to(device)
decoder_lstm = DecoderRNN(embed_size, hidden_size, vocab_size, num_layers, rnn_type="LSTM").to(device)

# 3. Otimizador e Loss
criterion = nn.CrossEntropyLoss(ignore_index=dataset.vocab.stoi["<PAD>"])
params = list(decoder_lstm.parameters()) + list(encoder.embed.parameters()) + list(encoder.bn.parameters())
optimizer = optim.Adam(params, lr=learning_rate)

# 4. Treinar
train_model(encoder, decoder_lstm, train_loader, criterion, optimizer, num_epochs, model_name="LSTM")

# 5. Salvar (Agora vai funcionar porque a variável existe!)
if not os.path.exists("modelos_salvos"):
    os.makedirs("modelos_salvos")

torch.save(encoder.state_dict(), "modelos_salvos/encoder.pth")
torch.save(decoder_lstm.state_dict(), "modelos_salvos/decoder_lstm.pth")
print("💾 Modelo LSTM salvo com sucesso!")

🤖 Criando modelo LSTM...
--- Iniciando Treino: LSTM ---
✅ Epoch 1 finalizada! Loss Média: 4.0664 | Tempo: 124s
✅ Epoch 2 finalizada! Loss Média: 3.3794 | Tempo: 123s
✅ Epoch 3 finalizada! Loss Média: 3.1491 | Tempo: 122s
✅ Epoch 4 finalizada! Loss Média: 2.9897 | Tempo: 123s
✅ Epoch 5 finalizada! Loss Média: 2.8778 | Tempo: 127s
✅ Epoch 6 finalizada! Loss Média: 2.7920 | Tempo: 127s
✅ Epoch 7 finalizada! Loss Média: 2.7239 | Tempo: 125s
✅ Epoch 8 finalizada! Loss Média: 2.6660 | Tempo: 126s
✅ Epoch 9 finalizada! Loss Média: 2.6165 | Tempo: 123s
✅ Epoch 10 finalizada! Loss Média: 2.5742 | Tempo: 125s
Treino de LSTM concluído!
💾 Modelo LSTM salvo com sucesso!


### GRU

In [ ]:
# --- MODELO 2: GRU ---

# 1. Instanciar Modelo
print("🤖 Criando modelo GRU...")
# Reutilize o Encoder, mas crie um novo Decoder
decoder_gru = DecoderRNN(embed_size, hidden_size, vocab_size, num_layers, rnn_type="GRU").to(device)

# 2. Otimizador e Loss
params_gru = list(decoder_gru.parameters()) + list(encoder.embed.parameters()) + list(encoder.bn.parameters())
optimizer_gru = optim.Adam(params_gru, lr=learning_rate)

# 3. Treinar
train_model(encoder, decoder_gru, train_loader, criterion, optimizer_gru, num_epochs, model_name="GRU")

# 4. Salvar
torch.save(decoder_gru.state_dict(), "modelos_salvos/decoder_gru.pth")
print("💾 Modelo GRU salvo com sucesso!")

🤖 Criando modelo GRU...
--- Iniciando Treino: GRU ---
✅ Epoch 1 finalizada! Loss Média: 3.9497 | Tempo: 124s
✅ Epoch 2 finalizada! Loss Média: 3.2578 | Tempo: 124s
✅ Epoch 3 finalizada! Loss Média: 3.0373 | Tempo: 125s
✅ Epoch 4 finalizada! Loss Média: 2.8979 | Tempo: 123s
✅ Epoch 5 finalizada! Loss Média: 2.7990 | Tempo: 124s
✅ Epoch 6 finalizada! Loss Média: 2.7259 | Tempo: 124s
✅ Epoch 7 finalizada! Loss Média: 2.6665 | Tempo: 124s
✅ Epoch 8 finalizada! Loss Média: 2.6173 | Tempo: 123s
✅ Epoch 9 finalizada! Loss Média: 2.5736 | Tempo: 123s
✅ Epoch 10 finalizada! Loss Média: 2.5389 | Tempo: 123s
Treino de GRU concluído!
💾 Modelo GRU salvo com sucesso!


### Transformer

In [ ]:
# --- MODELO 3: TRANSFORMER ---

# 1. Configurações Específicas
num_heads = 4 # Número de cabeças de atenção (pode ser ajustado)
num_layers_trans = 1 # Número de camadas do decoder (pode ser ajustado)

# 2. Instanciar Modelo
print("🤖 Criando modelo Transformer...")
# Reutilize o Encoder, mas crie um novo Decoder
decoder_transformer = DecoderTransformer(embed_size, vocab_size, num_heads, num_layers_trans).to(device)

# 3. Otimizador e Loss
params_trans = list(decoder_transformer.parameters()) + list(encoder.embed.parameters()) + list(encoder.bn.parameters())
optimizer_trans = optim.Adam(params_trans, lr=learning_rate)

# 4. Treinar
train_model(encoder, decoder_transformer, train_loader, criterion, optimizer_trans, num_epochs, model_name="Transformer")

# 5. Salvar
torch.save(decoder_transformer.state_dict(), "modelos_salvos/decoder_transformer.pth")
print("💾 Modelo Transformer salvo com sucesso!")

🤖 Criando modelo Transformer...
--- Iniciando Treino: Transformer ---
✅ Epoch 1 finalizada! Loss Média: 3.7727 | Tempo: 123s
✅ Epoch 2 finalizada! Loss Média: 3.1594 | Tempo: 126s
✅ Epoch 3 finalizada! Loss Média: 2.9584 | Tempo: 125s
✅ Epoch 4 finalizada! Loss Média: 2.8360 | Tempo: 124s
✅ Epoch 5 finalizada! Loss Média: 2.7447 | Tempo: 125s
✅ Epoch 6 finalizada! Loss Média: 2.6709 | Tempo: 124s
✅ Epoch 7 finalizada! Loss Média: 2.6117 | Tempo: 124s
✅ Epoch 8 finalizada! Loss Média: 2.5606 | Tempo: 126s
✅ Epoch 9 finalizada! Loss Média: 2.5149 | Tempo: 123s
✅ Epoch 10 finalizada! Loss Média: 2.4747 | Tempo: 126s
Treino de Transformer concluído!
💾 Modelo Transformer salvo com sucesso!


## 4. Geração e avaliação

Geração gulosa (até 20 tokens) e BLEU-4 do TorchMetrics em 150 lotes (4.800 pares imagem-legenda) do `train_loader`, com uma legenda de referência por imagem. As limitações dessa avaliação estão no [README](../README.md).

In [ ]:
# Função para gerar legenda com LSTM ou GRU
def generate_caption_rnn(encoder, decoder, image, vocab, max_len=20):
    encoder.eval()
    decoder.eval()

    with torch.no_grad():
        # 1. Extrair features da imagem
        features = encoder(image.unsqueeze(0).to(device))

        # 2. Preparar estado inicial
        states = None

        # A entrada inicial é a imagem
        inputs = features.unsqueeze(1)

        caption = []

        for _ in range(max_len):
            # Passar pela RNN
            hiddens, states = decoder.rnn(inputs, states)

            # Prever a próxima palavra
            outputs = decoder.linear(hiddens.squeeze(1))
            predicted = outputs.argmax(1)

            if predicted.item() == vocab.stoi["<EOS>"]:
                break

            # Adiciona a palavra na lista
            caption.append(vocab.itos[predicted.item()])

            # A próxima entrada é a palavra que acabamos de prever
            inputs = decoder.embed(predicted).unsqueeze(1)

    return caption

# Função para gerar legenda com Transformer
def generate_caption_transformer(encoder, decoder, image, vocab, max_len=20):
    encoder.eval()
    decoder.eval()

    with torch.no_grad():
        features = encoder(image.unsqueeze(0).to(device)).unsqueeze(1) # Memory

        # Começa com <SOS>
        indices = [vocab.stoi["<SOS>"]]

        for _ in range(max_len):
            tgt = torch.tensor(indices).unsqueeze(0).to(device)

            # No Transformer, passamos a sequência inteira gerada até agora
            # Ele vai prever a próxima palavra baseada em tudo que viu
            tgt_mask = decoder.generate_mask(tgt.size(1))

            embeddings = decoder.dropout(decoder.pos_encoder(decoder.embed(tgt)))
            output = decoder.transformer_decoder(tgt=embeddings, memory=features, tgt_mask=tgt_mask)
            output = decoder.linear(output[:, -1, :])

            predicted = output.argmax(1).item()

            if predicted == vocab.stoi["<EOS>"]:
                break

            indices.append(predicted)

    return [vocab.itos[idx] for idx in indices[1:]]

In [ ]:
def evaluate_model(encoder, decoder, data_loader, model_type="RNN", num_batches=5):
    bleu = BLEUScore()
    encoder.eval()
    decoder.eval()

    predicted_sentences = []
    actual_sentences = []

    print(f"Calculando BLEU para {model_type} (Aguarde, processando {num_batches} lotes)...")

    with torch.no_grad():
        for idx, (imgs, captions) in enumerate(data_loader):
            if idx >= num_batches: break # Avalia apenas alguns lotes para não demorar horas

            imgs = imgs.to(device)

            for i in range(imgs.size(0)):
                # 1. Gerar legenda predita
                if model_type == "Transformer":
                    prediction = generate_caption_transformer(encoder, decoder, imgs[i], dataset.vocab)
                else:
                    prediction = generate_caption_rnn(encoder, decoder, imgs[i], dataset.vocab)

                # Juntar lista de palavras em frase
                pred_str = " ".join(prediction)
                predicted_sentences.append(pred_str)

                # 2. Pegar legenda real
                caption_indices = captions[i].tolist()
                real_caption = [dataset.vocab.itos[idx] for idx in caption_indices if idx not in [0, 1, 2, 3]]
                actual_sentences.append([" ".join(real_caption)]) # O BLEU espera uma lista de referências

    # Calcular Score
    score = bleu(predicted_sentences, actual_sentences)
    return score.item()

In [ ]:
# --- Executar Avaliação para todos os modelos ---
num_batches_avaliacao = 150

# 1. Avaliar LSTM
score_lstm = evaluate_model(encoder, decoder_lstm, train_loader, model_type="RNN", num_batches=num_batches_avaliacao)
print(f"📊 BLEU Score LSTM: {score_lstm * 100:.2f}%")

# 2. Avaliar GRU
score_gru = evaluate_model(encoder, decoder_gru, train_loader, model_type="RNN", num_batches=num_batches_avaliacao)
print(f"📊 BLEU Score GRU: {score_gru * 100:.2f}%")

# 3. Avaliar Transformer
score_trans = evaluate_model(encoder, decoder_transformer, train_loader, model_type="Transformer", num_batches=num_batches_avaliacao)
print(f"📊 BLEU Score Transformer: {score_trans * 100:.2f}%")

Calculando BLEU para RNN (Aguarde, processando 150 lotes)...
📊 BLEU Score LSTM: 2.21%
Calculando BLEU para RNN (Aguarde, processando 150 lotes)...
📊 BLEU Score GRU: 1.70%
Calculando BLEU para Transformer (Aguarde, processando 150 lotes)...
📊 BLEU Score Transformer: 8.48%


## Resultados

A comparação com o encoder ViT-B/16 e a análise estão no [README](../README.md).